# 01 — Data audit

What the dataset is, where it comes from, and why this region of the Moon.

LunaRec decides which candidate lunar patches receive scarce high-resolution
imaging. Before any of that is meaningful, this notebook establishes that the
data is real, the region is justified, and the features behave.

Four Chandrayaan-2 OHRC scenes across two geographic blocks at 60–70°S.

**Cell 1** loads the four patch files and audits them. It is deliberately a
check, not a summary: row counts, extents and hazard means are compared against
the values the recorded results were produced from. A mismatch here means the
patch files were rebuilt with different parameters, and nothing downstream
would be comparable to `results/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the project root whether this runs from notebooks/ or the root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PATCHES = ROOT / "data" / "patches" / "current"

SCENES = {
    "0858":  ("B", "patches_0858_rt.csv"),
    "1647a": ("B", "patches_1647a_rt.csv"),
    "C1":    ("C", "patches_C1.csv"),
    "C2":    ("C", "patches_C2.csv"),
}

# The three attributes the recommender ranks on, all OHRC-derived.
# DEM attributes are excluded; see docs/scene_mapping.md for why.
ATTRS = ["shadow_local", "dn_mean", "dn_std"]
REQUIRED = ["patch_x", "patch_y"] + ATTRS

scenes = {}
rows = []
for label, (block, fname) in SCENES.items():
    path = PATCHES / fname
    if not path.exists():
        raise FileNotFoundError(f"{path} -- check data/patches/current/")
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"{fname} is missing {missing}")
    scenes[label] = df
    rows.append({
        "scene": label,
        "block": block,
        "patches": len(df),
        "cols": df.shape[1],
        "x_min_km": df.patch_x.min() / 1000,
        "x_max_km": df.patch_x.max() / 1000,
        "y_min_km": df.patch_y.min() / 1000,
        "y_max_km": df.patch_y.max() / 1000,
        "hazard_mean": df.shadow_local.mean(),
        "nan_rows": int(df[REQUIRED].isna().any(axis=1).sum()),
    })

audit = pd.DataFrame(rows).set_index("scene")
print(f"project root: {ROOT}")
print(f"total patches: {audit.patches.sum()}")
print()
audit.round(3)